## Câu 1: PHÂN LOẠI HOA IRIS

### 1. Nhập thư viện

In [5]:
import pandas as pd
import numpy as np
import math
import random

### 2. Định nghĩa lớp Naive Bayes (Từ file Naive_Bayes.py)

In [6]:
# Ô 2: Định nghĩa lớp Naive Bayes (ĐÃ SỬA LỖI CHO PANDAS MỚI)
class Naive_Bayes:
  def __init__(self, data_set):
    self.ds = data_set
    # Tính mean và variance cho từng đặc trưng (feature) theo từng lớp (class)
    # Cột 4 là cột nhãn (Species)
    self.ds_means = self.ds.groupby(4).mean()
    self.ds_variances = self.ds.groupby(4).var()
    # Tính xác suất tiên nghiệm (prior probability) cho từng lớp
    self.class_probabilities = self.get_class_probabilities(self.ds)

  def get_class_probabilities(self, data_set):
    class_sizes = data_set.groupby(4).size()
    ds_total = data_set.shape[0]
    probs = {}
    
    # --- SỬA LỖI TẠI ĐÂY: dùng .items() thay vì .iteritems() ---
    for i in class_sizes.items(): 
      probs[i[0]] = i[1] / ds_total
    return probs

  def get_probability_density(self, x, mean, variance):
    # Hàm mật độ xác suất Gaussian
    # Tính xác suất P(x | class)
    pd = 1 / (np.sqrt(2 * np.pi * variance)) * np.exp((-(x - mean)**2) / (2 * variance))
    return pd

  def predict(self, x):
    feature_class_probabilities = {}
    for group, class_prob in self.class_probabilities.items():
      feature_class_probabilities[group] = class_prob
      for i in range(len(x)):
        # Nhân xác suất của từng thuộc tính (Naive assumption)
        feature_class_probabilities[group] *= self.get_probability_density(x[i], self.ds_means.loc[group][i], self.ds_variances.loc[group][i])
    
    # Chọn lớp có xác suất cao nhất
    feature_class = max(feature_class_probabilities, key=feature_class_probabilities.get)
    return feature_class

  def test(self, test_data):
    correct = 0
    total = 0
    for row in test_data.itertuples():
      feature_set = row[1:5] # Lấy 4 thuộc tính (bỏ cột index 0 do itertuples sinh ra)
      # Lưu ý: row[0] là index của dataframe, row[1]..row[4] là feature, row[5] là label
      # Trong dataframe (0,1,2,3,4), itertuples sẽ trả về (Index, 0, 1, 2, 3, 4)
      # Nên feature là row[1] đến row[4], label là row[5]
      
      group = self.predict(feature_set)
      
      if group == row[5]: 
        correct += 1
      else:
        print(f"  Dữ liệu: {feature_set} -> Dự đoán: {group} | Thực tế: {row[5]}")
      total += 1
    accuracy = correct / total
    print(f"\nTổng số mẫu kiểm thử: {total}")
    print(f"Số dự đoán đúng: {correct}")
    print(f"==> Độ chính xác: {accuracy * 100:.2f}%")

### 3. Tải và chuẩn bị dữ liệu

In [7]:
# Đọc dữ liệu từ file Iris.csv
try:
    df = pd.read_csv("Iris.csv")
except FileNotFoundError:
    print("LỖI: Không tìm thấy file 'Iris.csv'.")
    print("Hãy đảm bảo file 'Iris.csv' nằm cùng thư mục với notebook này.")
    # Dừng ở đây nếu lỗi
    raise

# Xóa cột 'Id' không cần thiết
if 'Id' in df.columns:
    df.drop(['Id'], axis=1, inplace=True)

# Chuyển DataFrame sang list để xáo trộn
data_set = df.values.tolist()

# Xáo trộn dữ liệu
random.seed(42) # Đặt seed để kết quả cố định
random.shuffle(data_set)

# Chia dữ liệu: 120 mẫu train, 30 mẫu test
train_data_list = data_set[:120]
test_data_list = data_set[120:]

# Chuyển lại thành DataFrame theo cấu trúc mà class Naive_Bayes yêu cầu
# Class của bạn yêu cầu cột nhãn ở index 4
train_data = pd.DataFrame(train_data_list, columns=[0, 1, 2, 3, 4])
test_data = pd.DataFrame(test_data_list, columns=[0, 1, 2, 3, 4])

print(f"Đã tải và chia dữ liệu:")
print(f" - Tập huấn luyện: {len(train_data)} mẫu")
print(f" - Tập kiểm thử: {len(test_data)} mẫu")

Đã tải và chia dữ liệu:
 - Tập huấn luyện: 120 mẫu
 - Tập kiểm thử: 30 mẫu


### 4. Huấn luyện và chạy kiểm thử

In [8]:
print("Bắt đầu huấn luyện...")
# 1. Huấn luyện: Khởi tạo class với dữ liệu train
# (Class của bạn thực hiện toàn bộ việc tính toán trong __init__)
nb = Naive_Bayes(train_data)

print("Huấn luyện hoàn tất. Bắt đầu kiểm thử...")
print("Các mẫu dự đoán sai (nếu có):")

# 2. Kiểm thử: Gọi hàm test với dữ liệu test
nb.test(test_data)

Bắt đầu huấn luyện...
Huấn luyện hoàn tất. Bắt đầu kiểm thử...
Các mẫu dự đoán sai (nếu có):
  Dữ liệu: (5.9, 3.2, 4.8, 1.8) -> Dự đoán: Iris-virginica | Thực tế: Iris-versicolor

Tổng số mẫu kiểm thử: 30
Số dự đoán đúng: 29
==> Độ chính xác: 96.67%


## CÂU 2: NHẬN DẠNG KÝ TỰ (LETTER RECOGNITION)

### 1. Nhập thư viện và nhập dữ liệu

In [81]:
# Nhập thư viện
import pandas as pd
import numpy as np
import math

In [82]:
# Đọc dữ liệu
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/letter-recognition/letter-recognition.data"
cols = ['letter', 'x-box', 'y-box', 'width', 'high', 'onpix', 'x-bar', 'y-bar',
        'x2bar', 'y2bar', 'xybar', 'x2ybr', 'xy2br', 'x-ege', 'xegvy', 'y-ege', 'yegvx']
data = pd.read_csv(url, header=None, names=cols)


In [83]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 17 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   letter  20000 non-null  object
 1   x-box   20000 non-null  int64 
 2   y-box   20000 non-null  int64 
 3   width   20000 non-null  int64 
 4   high    20000 non-null  int64 
 5   onpix   20000 non-null  int64 
 6   x-bar   20000 non-null  int64 
 7   y-bar   20000 non-null  int64 
 8   x2bar   20000 non-null  int64 
 9   y2bar   20000 non-null  int64 
 10  xybar   20000 non-null  int64 
 11  x2ybr   20000 non-null  int64 
 12  xy2br   20000 non-null  int64 
 13  x-ege   20000 non-null  int64 
 14  xegvy   20000 non-null  int64 
 15  y-ege   20000 non-null  int64 
 16  yegvx   20000 non-null  int64 
dtypes: int64(16), object(1)
memory usage: 2.6+ MB


In [84]:
print(data.head())

  letter  x-box  y-box  width  high  onpix  x-bar  y-bar  x2bar  y2bar  xybar  \
0      T      2      8      3     5      1      8     13      0      6      6   
1      I      5     12      3     7      2     10      5      5      4     13   
2      D      4     11      6     8      6     10      6      2      6     10   
3      N      7     11      6     6      3      5      9      4      6      4   
4      G      2      1      3     1      1      8      6      6      6      6   

   x2ybr  xy2br  x-ege  xegvy  y-ege  yegvx  
0     10      8      0      8      0      8  
1      3      9      2      8      4     10  
2      3      7      3      7      3      9  
3      4     10      6     10      2      8  
4      5      9      1      7      5     10  


### 2. Chuyển kí tự sang dạng số

In [85]:
# Chuyển nhãn sang số (A=0, B=1, ..., Z=25)
letters = sorted(data['letter'].unique())
label_to_int = {l:i for i, l in enumerate(letters)} # biến nhãn chữ thành số 0..25
int_to_label = {i:l for l, i in label_to_int.items()} # đảo ngược dùng khi in kết quả.
data['label'] = data['letter'].map(label_to_int)
data = data.drop(columns=['letter'])

### 3. Chia dữ liệu train/test

In [86]:
# Chia dữ liệu train/test 70/30
np.random.seed(42)
indices = np.random.permutation(len(data))
train_size = int(0.7 * len(data))
train_idx, test_idx = indices[:train_size], indices[train_size:]

train = data.iloc[train_idx]
test  = data.iloc[test_idx]

X_train, y_train = train.drop(columns=['label']).values, train['label'].values
X_test, y_test   = test.drop(columns=['label']).values, test['label'].values

In [87]:
print(f"Train shape: {len(X_train)} x {len(X_train[0])}")
print(f"Test shape:  {len(X_test)} x {len(X_test[0])}")

Train shape: 14000 x 16
Test shape:  6000 x 16


### 4. Huấn luyện Naive Bayes

#### 4.1 Tính trung bình, phương sai và prior

In [88]:
# Huấn luyện Naive Bayes
classes = np.unique(y_train)

# Tính trung bình, phương sai và prior cho mỗi lớp
mean = {}
var  = {}
prior = {}

for c in classes:
    X_c = X_train[y_train == c]
    mean[c] = X_c.mean(axis=0)
    var[c]  = X_c.var(axis=0) + 1e-6   # tránh chia cho 0
    prior[c] = len(X_c) / len(X_train)

#### 4.2 Dự đoán

In [89]:
# Dự đoán
# Trả về đúng giá trị log-likelihood
def likelihood(x, mean, var):
    # Tính log xác suất để tránh tràn số
    return -0.5 * np.sum(np.log(2.0 * np.pi * var)) - 0.5 * np.sum(((x - mean)**2) / var)

def predict(x):
    posteriors = []
    for c in classes:
        log_prior = math.log(prior[c])
        log_likelihood = likelihood(x, mean[c], var[c])
        posterior = log_prior + log_likelihood
        posteriors.append(posterior)
    return classes[np.argmax(posteriors)]

#### 4.3 Kết quả

In [90]:
# Kiểm thử mô hình
predictions = [predict(x) for x in X_test]
accuracy = np.mean(np.array(predictions) == y_test)
print(f"Độ chính xác: {accuracy * 100:.2f}%")

Độ chính xác: 64.48%


In [91]:
#  Hiển thị vài kết quả mẫu
for i in range(15):
    true_label = int_to_label[y_test[i]]
    pred_label = int_to_label[predictions[i]]
    print(f"Mẫu {i+1}: Thật = {true_label}, Dự đoán = {pred_label}")

Mẫu 1: Thật = P, Dự đoán = P
Mẫu 2: Thật = Y, Dự đoán = W
Mẫu 3: Thật = H, Dự đoán = Q
Mẫu 4: Thật = M, Dự đoán = M
Mẫu 5: Thật = X, Dự đoán = X
Mẫu 6: Thật = X, Dự đoán = B
Mẫu 7: Thật = I, Dự đoán = I
Mẫu 8: Thật = T, Dự đoán = T
Mẫu 9: Thật = A, Dự đoán = A
Mẫu 10: Thật = V, Dự đoán = V
Mẫu 11: Thật = S, Dự đoán = I
Mẫu 12: Thật = D, Dự đoán = D
Mẫu 13: Thật = K, Dự đoán = C
Mẫu 14: Thật = C, Dự đoán = C
Mẫu 15: Thật = E, Dự đoán = E
